In [ ]:
!pip install requests beautifulsoup4 feedparser tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 1.6 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=680b5d5a611230f9c78fce21eff8f8e60b8e914496071641fea6f1ff08ef2b77
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
import re
import pickle
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, SpatialDropout1D,
    Conv1D, MaxPooling1D,
    Bidirectional, LSTM,
    Dense, Dropout,
    GlobalMaxPooling1D,
    Concatenate
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def get_callbacks(patience=3):
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
    ]

Model 1 - Clickbait Detection

In [ ]:
df_clickbait = pd.read_csv("final-clickbait.csv")

df_clickbait = df_clickbait.dropna(
    subset=["full_text", "label"]
)

df_clickbait["full_text"] = df_clickbait["full_text"].astype(str)
df_clickbait["label"] = df_clickbait["label"].astype(int)

df_clickbait["clean_text"] = df_clickbait["full_text"].apply(clean_text)

In [ ]:
X_cb = df_clickbait["clean_text"]
y_cb = df_clickbait["label"]

X_cb_train, X_cb_temp, y_cb_train, y_cb_temp = train_test_split(
    X_cb, y_cb,
    test_size=0.2,
    random_state=42,
    stratify=y_cb
)

X_cb_val, X_cb_test, y_cb_val, y_cb_test = train_test_split(
    X_cb_temp, y_cb_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_cb_temp
)

In [ ]:
MAX_WORDS_CLICKBAIT = 10000
MAX_LEN_CLICKBAIT = 40

tokenizer_clickbait = Tokenizer(
    num_words=MAX_WORDS_CLICKBAIT,
    oov_token="<OOV>"
)

tokenizer_clickbait.fit_on_texts(X_cb_train)

X_cb_train_pad = pad_sequences(
    tokenizer_clickbait.texts_to_sequences(X_cb_train),
    maxlen=MAX_LEN_CLICKBAIT,
    padding="post",
    truncating="post"
)

X_cb_val_pad = pad_sequences(
    tokenizer_clickbait.texts_to_sequences(X_cb_val),
    maxlen=MAX_LEN_CLICKBAIT,
    padding="post",
    truncating="post"
)

X_cb_test_pad = pad_sequences(
    tokenizer_clickbait.texts_to_sequences(X_cb_test),
    maxlen=MAX_LEN_CLICKBAIT,
    padding="post",
    truncating="post"
)

In [ ]:
clickbait_input = Input(
    shape=(MAX_LEN_CLICKBAIT,),
    name="clickbait_input"
)

x = Embedding(
    input_dim=MAX_WORDS_CLICKBAIT,
    output_dim=128,
    name="embedding"
)(clickbait_input)

x = SpatialDropout1D(
    0.3,
    name="spatial_dropout"
)(x)

x = Conv1D(
    filters=128,
    kernel_size=3,
    activation="relu",
    padding="same",
    name="conv1d"
)(x)

x = MaxPooling1D(
    pool_size=2,
    name="maxpool"
)(x)

x = Bidirectional(
    LSTM(
        64,
        dropout=0.3,
        recurrent_dropout=0.3
    ),
    name="bilstm"
)(x)

x = Dense(
    64,
    activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(0.001),
    name="dense_1"
)(x)

x = Dropout(
    0.5,
    name="dropout"
)(x)

clickbait_output = Dense(
    1,
    activation="sigmoid",
    name="output"
)(x)


In [ ]:
model_clickbait = Model(
    inputs=clickbait_input,
    outputs=clickbait_output,
    name="functional_clickbait_model"
)
model_clickbait.summary()

Model: "functional_clickbait_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ clickbait_input (InputLayer)    │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 40, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout                 │ (None, 40, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 40, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool (MaxPooling1D)          │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,436,417 (5.48 MB)

 Trainable params: 1,436,417 (5.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy()

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_cb_train_pad, y_cb_train.values)
).shuffle(1000).batch(32)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_cb_val_pad, y_cb_val.values)
).batch(32)

EPOCHS = 15
PATIENCE = 3

best_val_loss = float("inf")
wait = 0
best_weights = None

history_clickbait = {
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": []
}

for epoch in range(EPOCHS):
    train_loss = tf.keras.metrics.Mean()
    train_acc = tf.keras.metrics.BinaryAccuracy()
    train_precision = tf.keras.metrics.Precision()
    train_recall = tf.keras.metrics.Recall()

    for x_batch, y_batch in train_dataset:
        y_batch = tf.cast(
            tf.reshape(y_batch, (-1, 1)),
            tf.float32
        )

        with tf.GradientTape() as tape:
            y_pred = model_clickbait(
                x_batch,
                training=True
            )

            loss = loss_fn(
                y_batch,
                y_pred
            )

            reg_loss = tf.add_n(
                model_clickbait.losses
            ) if model_clickbait.losses else 0

            total_loss = loss + reg_loss

        gradients = tape.gradient(
            total_loss,
            model_clickbait.trainable_variables
        )

        optimizer.apply_gradients(
            zip(
                gradients,
                model_clickbait.trainable_variables
            )
        )

        train_loss.update_state(total_loss)
        train_acc.update_state(y_batch, y_pred)
        train_precision.update_state(y_batch, y_pred)
        train_recall.update_state(y_batch, y_pred)

    val_loss = tf.keras.metrics.Mean()
    val_acc = tf.keras.metrics.BinaryAccuracy()
    val_precision = tf.keras.metrics.Precision()
    val_recall = tf.keras.metrics.Recall()

    for x_val_batch, y_val_batch in val_dataset:
        y_val_batch = tf.cast(
            tf.reshape(y_val_batch, (-1, 1)),
            tf.float32
        )

        y_val_pred = model_clickbait(
            x_val_batch,
            training=False
        )

        v_loss = loss_fn(
            y_val_batch,
            y_val_pred
        )

        reg_loss = tf.add_n(
            model_clickbait.losses
        ) if model_clickbait.losses else 0

        total_v_loss = v_loss + reg_loss

        val_loss.update_state(total_v_loss)
        val_acc.update_state(y_val_batch, y_val_pred)
        val_precision.update_state(y_val_batch, y_val_pred)
        val_recall.update_state(y_val_batch, y_val_pred)

    history_clickbait["accuracy"].append(float(train_acc.result()))
    history_clickbait["loss"].append(float(train_loss.result()))
    history_clickbait["precision"].append(float(train_precision.result()))
    history_clickbait["recall"].append(float(train_recall.result()))
    history_clickbait["val_accuracy"].append(float(val_acc.result()))
    history_clickbait["val_loss"].append(float(val_loss.result()))
    history_clickbait["val_precision"].append(float(val_precision.result()))
    history_clickbait["val_recall"].append(float(val_recall.result()))

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - "
        f"accuracy: {train_acc.result():.4f} - "
        f"loss: {train_loss.result():.4f} - "
        f"precision: {train_precision.result():.4f} - "
        f"recall: {train_recall.result():.4f} - "
        f"val_accuracy: {val_acc.result():.4f} - "
        f"val_loss: {val_loss.result():.4f} - "
        f"val_precision: {val_precision.result():.4f} - "
        f"val_recall: {val_recall.result():.4f}"
    )

    if val_loss.result() < best_val_loss:
        best_val_loss = float(val_loss.result())
        best_weights = model_clickbait.get_weights()
        wait = 0
    else:
        wait += 1

    if wait >= PATIENCE:
        print("Early stopping aktif.")
        break

if best_weights is not None:
    model_clickbait.set_weights(best_weights)

Epoch 1/15 - accuracy: 0.7085 - loss: 0.5481 - precision: 0.7016 - recall: 0.5175 - val_accuracy: 0.7366 - val_loss: 0.5201 - val_precision: 0.7113 - val_recall: 0.6759
Epoch 2/15 - accuracy: 0.8763 - loss: 0.3388 - precision: 0.8552 - recall: 0.8047 - val_accuracy: 0.8328 - val_loss: 0.4393 - val_precision: 0.8043 - val_recall: 0.7989
Epoch 3/15 - accuracy: 0.9262 - loss: 0.2116 - precision: 0.9211 - recall: 0.9183 - val_accuracy: 0.8651 - val_loss: 0.3551 - val_precision: 0.8692 - val_recall: 0.8543
Epoch 4/15 - accuracy: 0.9749 - loss: 0.0911 - precision: 0.9726 - recall: 0.9591 - val_accuracy: 0.8678 - val_loss: 0.3524 - val_precision: 0.8529 - val_recall: 0.8512
Epoch 5/15 - accuracy: 0.9801 - loss: 0.0779 - precision: 0.9789 - recall: 0.9701 - val_accuracy: 0.8823 - val_loss: 0.2919 - val_precision: 0.8791 - val_recall: 0.8757
Early stopping aktif.


In [ ]:
y_cb_pred_prob = model_clickbait.predict(X_cb_test_pad,verbose=0)

THRESHOLD_CLICKBAIT = 0.45

y_cb_pred = (
    y_cb_pred_prob >= THRESHOLD_CLICKBAIT
).astype(int).flatten()

print("Accuracy:", accuracy_score(y_cb_test, y_cb_pred))

print(classification_report(
    y_cb_test,
    y_cb_pred,
    target_names=["non-clickbait", "clickbait"]
))

print(confusion_matrix(y_cb_test, y_cb_pred))

Accuracy: 0.8783967391304348
               precision    recall  f1-score   support

non-clickbait       0.89      0.90      0.90       856
    clickbait       0.86      0.85      0.85       616

     accuracy                           0.88      1472
    macro avg       0.88      0.87      0.87      1472
 weighted avg       0.88      0.88      0.88      1472

[[772  84]
 [ 95 521]]


In [ ]:
model_clickbait.save("model_clickbait_final.keras")

with open("tokenizer_clickbait.pkl", "wb") as f:
    pickle.dump(tokenizer_clickbait, f)

Model 2 - Stance Detection

In [ ]:
df_stance = pd.read_csv("final_data_stance.csv")

df_stance = df_stance.dropna(
    subset=["claim", "article", "label"]
)

df_stance["claim"] = (
    df_stance["claim"]
    .astype(str)
    .apply(clean_text)
)

df_stance["article"] = (
    df_stance["article"]
    .astype(str)
    .apply(clean_text)
)

df_stance["label"] = df_stance["label"].astype(int)

claims = df_stance["claim"]
articles = df_stance["article"]
labels = df_stance["label"]

In [ ]:
claim_train, claim_test, article_train, article_test, y_train, y_test = train_test_split(
    claims,
    articles,
    labels,
    test_size=0.3,
    stratify=labels,
    random_state=42
)

In [ ]:
MAX_WORDS_STANCE = 20000
MAX_CLAIM_LEN = 40
MAX_ARTICLE_LEN = 200

claim_tokenizer = Tokenizer(
    num_words=MAX_WORDS_STANCE,
    oov_token="<OOV>"
)

article_tokenizer = Tokenizer(
    num_words=MAX_WORDS_STANCE,
    oov_token="<OOV>"
)

claim_tokenizer.fit_on_texts(claim_train)
article_tokenizer.fit_on_texts(article_train)

X_claim_train = pad_sequences(
    claim_tokenizer.texts_to_sequences(claim_train),
    maxlen=MAX_CLAIM_LEN,
    padding="post",
    truncating="post"
)

X_claim_test = pad_sequences(
    claim_tokenizer.texts_to_sequences(claim_test),
    maxlen=MAX_CLAIM_LEN,
    padding="post",
    truncating="post"
)

X_article_train = pad_sequences(
    article_tokenizer.texts_to_sequences(article_train),
    maxlen=MAX_ARTICLE_LEN,
    padding="post",
    truncating="post"
)

X_article_test = pad_sequences(
    article_tokenizer.texts_to_sequences(article_test),
    maxlen=MAX_ARTICLE_LEN,
    padding="post",
    truncating="post"
)

In [ ]:
claim_input = Input(
    shape=(MAX_CLAIM_LEN,),
    name="claim_input"
)

x1 = Embedding(
    input_dim=MAX_WORDS_STANCE,
    output_dim=128,
    name="claim_embedding"
)(claim_input)

x1 = Bidirectional(
    LSTM(
        64,
        return_sequences=True
    ),
    name="claim_bilstm"
)(x1)

x1 = GlobalMaxPooling1D(
    name="claim_pooling"
)(x1)

article_input = Input(
    shape=(MAX_ARTICLE_LEN,),
    name="article_input"
)

x2 = Embedding(
    input_dim=MAX_WORDS_STANCE,
    output_dim=128,
    name="article_embedding"
)(article_input)

x2 = Bidirectional(
    LSTM(
        32,
        return_sequences=True,
        dropout=0.3,
        recurrent_dropout=0.3
    ),
    name="article_bilstm"
)(x2)

x2 = GlobalMaxPooling1D(
    name="article_pooling"
)(x2)

combined = Concatenate(
    name="concatenate"
)([x1, x2])

x = Dense(
    64,
    activation="relu",
    name="dense_1"
)(combined)

x = Dropout(
    0.5,
    name="dropout_1"
)(x)

x = Dense(
    64,
    activation="relu",
    name="dense_2"
)(x)

stance_output = Dense(
    1,
    activation="sigmoid",
    name="output"
)(x)

In [ ]:
model_stance = Model(
    inputs=[claim_input, article_input],
    outputs=stance_output,
    name="functional_stance_model"
)

model_stance.summary()

Model: "functional_stance_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ claim_input         │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ article_input       │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ claim_embedding     │ (None, 40, 128)   │  2,560,000 │ claim_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ article_embedding   │ (None, 200, 128)  │  2,560,000 │ article_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ claim_bilstm        │ (None, 40, 128)   │     98,816 │ claim_embedding[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ article_bilstm      │ (None, 200, 64)   │     41,216 │ article_embeddin… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ claim_pooling       │ (None, 128)       │          0 │ claim_bilstm[0][… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ article_pooling     │ (None, 64)        │          0 │ article_bilstm[0… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ claim_pooling[0]… │
│ (Concatenate)       │                   │            │ article_pooling[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │     12,352 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      4,160 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         65 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,276,609 (20.13 MB)

 Trainable params: 5,276,609 (20.13 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
X_claim_train_final, X_claim_val, X_article_train_final, X_article_val, y_train_final, y_val = train_test_split(
    X_claim_train,
    X_article_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

loss_fn = tf.keras.losses.BinaryCrossentropy()

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        {
            "claim_input": X_claim_train_final,
            "article_input": X_article_train_final
        },
        y_train_final.values
    )
).shuffle(1000).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        {
            "claim_input": X_claim_val,
            "article_input": X_article_val
        },
        y_val.values
    )
).batch(16)

EPOCHS = 10
PATIENCE = 2

best_val_loss = float("inf")
wait = 0
best_weights = None

history_stance = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):
    train_loss = tf.keras.metrics.Mean()
    train_acc = tf.keras.metrics.BinaryAccuracy()

    for x_batch, y_batch in train_dataset:
        y_batch = tf.cast(
            tf.reshape(y_batch, (-1, 1)),
            tf.float32
        )

        with tf.GradientTape() as tape:
            y_pred = model_stance(
                x_batch,
                training=True
            )

            loss = loss_fn(
                y_batch,
                y_pred
            )

            reg_loss = tf.add_n(
                model_stance.losses
            ) if model_stance.losses else 0

            total_loss = loss + reg_loss

        gradients = tape.gradient(
            total_loss,
            model_stance.trainable_variables
        )

        optimizer.apply_gradients(
            zip(
                gradients,
                model_stance.trainable_variables
            )
        )

        train_loss.update_state(total_loss)
        train_acc.update_state(y_batch, y_pred)

    val_loss = tf.keras.metrics.Mean()
    val_acc = tf.keras.metrics.BinaryAccuracy()

    for x_val_batch, y_val_batch in val_dataset:
        y_val_batch = tf.cast(
            tf.reshape(y_val_batch, (-1, 1)),
            tf.float32
        )

        y_val_pred = model_stance(
            x_val_batch,
            training=False
        )

        v_loss = loss_fn(
            y_val_batch,
            y_val_pred
        )

        reg_loss = tf.add_n(
            model_stance.losses
        ) if model_stance.losses else 0

        total_v_loss = v_loss + reg_loss

        val_loss.update_state(total_v_loss)
        val_acc.update_state(y_val_batch, y_val_pred)

    history_stance["accuracy"].append(float(train_acc.result()))
    history_stance["loss"].append(float(train_loss.result()))
    history_stance["val_accuracy"].append(float(val_acc.result()))
    history_stance["val_loss"].append(float(val_loss.result()))

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - "
        f"accuracy: {train_acc.result():.4f} - "
        f"loss: {train_loss.result():.4f} - "
        f"val_accuracy: {val_acc.result():.4f} - "
        f"val_loss: {val_loss.result():.4f}"
    )

    if val_loss.result() < best_val_loss:
        best_val_loss = float(val_loss.result())
        best_weights = model_stance.get_weights()
        wait = 0
    else:
        wait += 1

    if wait >= PATIENCE:
        print("Early stopping aktif.")
        break

if best_weights is not None:
    model_stance.set_weights(best_weights)

Epoch 1/15 - accuracy: 0.4976 - loss: 0.6921 - val_accuracy: 0.6635 - val_loss: 0.6811
Epoch 2/15 - accuracy: 0.6256 - loss: 0.6722 - val_accuracy: 0.6731 - val_loss: 0.6392
Epoch 3/15 - accuracy: 0.7343 - loss: 0.5735 - val_accuracy: 0.7500 - val_loss: 0.5145
Epoch 4/15 - accuracy: 0.8478 - loss: 0.4019 - val_accuracy: 0.8365 - val_loss: 0.4406
Epoch 5/15 - accuracy: 0.9203 - loss: 0.2245 - val_accuracy: 0.8604 - val_loss: 0.4012
Epoch 6/15 - accuracy: 0.9710 - loss: 0.0951 - val_accuracy: 0.8592 - val_loss: 0.4154
Early stopping aktif.


In [ ]:
stance_pred_prob = model_stance.predict(
    [X_claim_test, X_article_test], verbose=0
)

stance_pred = (
    stance_pred_prob >= 0.5
).astype(int).flatten()

print("Accuracy:", accuracy_score(y_test, stance_pred))

print(classification_report(
    y_test,
    stance_pred
))

print(confusion_matrix(
    y_test,
    stance_pred
))

Accuracy: 0.8586956521739131
              precision    recall  f1-score   support

   mendukung       0.91      0.82      0.87        51
   membantah       0.80      0.90      0.85        41

    accuracy                           0.86        92
   macro avg       0.86      0.86      0.86        92
weighted avg       0.86      0.86      0.86        92

[[42  9]
 [ 4 37]]


In [ ]:
model_stance.save("model_stance_functional.keras")

with open("claim_tokenizer.pkl", "wb") as f:
    pickle.dump(claim_tokenizer, f)

with open("article_tokenizer.pkl", "wb") as f:
    pickle.dump(article_tokenizer, f)

stance_config = {
    "MAX_CLAIM_LEN": MAX_CLAIM_LEN,
    "MAX_ARTICLE_LEN": MAX_ARTICLE_LEN,
    "MAX_WORDS_STANCE": MAX_WORDS_STANCE
}

with open("stance_config.pkl", "wb") as f:
    pickle.dump(stance_config, f)

inferensi

In [1]:
import re
import pickle
import requests
import pandas as pd
import tensorflow as tf
import json

from bs4 import BeautifulSoup
from urllib.parse import quote
from flask import Flask, request, jsonify

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [4]:
model_clickbait = tf.keras.models.load_model("model_clickbait_final.keras")
model_stance = tf.keras.models.load_model("model_stance_functional.keras")
pers = pd.read_csv("final-pers.csv")

with open("tokenizer_clickbait.pkl", "rb") as f:
    tokenizer_clickbait = pickle.load(f)

with open("claim_tokenizer.pkl", "rb") as f:
    claim_tokenizer = pickle.load(f)

with open("article_tokenizer.pkl", "rb") as f:
    article_tokenizer = pickle.load(f)

with open("stance_config.pkl", "rb") as f:
    stance_config = pickle.load(f)

with open("negatif-indonesia.txt", "r", encoding="utf-8") as f:
    negative_words = set(
        line.strip().lower()
        for line in f
        if line.strip()
    )

with open("positif-indonesia.txt", "r", encoding="utf-8") as f:
    positive_words = set(
        line.strip().lower()
        for line in f
        if line.strip()
    )

strong_bantah_keywords = [
    "hoaks",
    "hoax",
    "[hoaks]",
    "cek fakta",
    "dibantah",
    "klarifikasi",
    "tidak benar",
    "bukan fakta",
    "fakta sebenarnya",
    "fact check",
    "[hoaks]",
    "[salah]",
    "penipuan",
    "[penipuan]",
    "keliru"
]

weak_bantah_keywords = [
    "palsu",
    "bohong",
    "fitnah",
    "misinformasi",
    "disinformasi",
    "fake",
    "menyesatkan",
    "tidak terbukti",
    "tidak jadi",
    "tidak akan",
    "diragukan",
    "belum terbukti"
]

strong_dukung_keywords = [
    "dikonfirmasi",
    "terbukti",
    "dipastikan",
    "menegaskan",
    "didakwa",
    "dituntut",
    "putusan",
    "vonis",
    "terdakwa",
    "umumkan",
    "mengumumkan",
    "resmi",
    "sidang"
]

weak_dukung_keywords = [
    "benar",
    "valid",
    "official",
    "nyata",
    "jadi",
    "mulai berlaku",
    "berlangsung",
    "terjadi",
    "mengakui",
    "menyatakan",
    "mengonfirmasi",
    "pengadilan",
    "kejaksaan",
    "jaksa tuntut",
]

In [5]:
MAX_LEN_CLICKBAIT = 40

MAX_CLAIM_LEN = stance_config["MAX_CLAIM_LEN"]
MAX_ARTICLE_LEN = stance_config["MAX_ARTICLE_LEN"]

THRESHOLD_CLICKBAIT = 0.45

In [6]:
def search_google_news(headline, max_articles=10):
    query_encoded = quote(headline)

    rss_url = (
        "https://news.google.com/rss/search?"
        f"q={query_encoded}&hl=id&gl=ID&ceid=ID:id"
    )

    response = requests.get(
        rss_url,
        timeout=10,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    soup = BeautifulSoup(response.content, "xml")

    items = soup.find_all("item")[:max_articles]

    articles = []

    for item in items:
        title = item.title.text if item.title else ""
        link = item.link.text if item.link else ""
        source = item.source.text if item.source else ""
        published = item.pubDate.text if item.pubDate else ""

        description = item.description.text if item.description else ""
        snippet = BeautifulSoup(description, "html.parser").get_text(" ", strip=True)

        content = snippet if snippet else title

        articles.append({
            "title": title,
            "url": link,
            "source": source,
            "published": published,
            "content": content
        })

    return pd.DataFrame(articles)

In [7]:
def predict_clickbait(headline):
    clean = clean_text(headline)

    seq = tokenizer_clickbait.texts_to_sequences([clean])

    pad = pad_sequences(
        seq,
        maxlen=MAX_LEN_CLICKBAIT,
        padding="post",
        truncating="post"
    )

    score = model_clickbait.predict(pad, verbose=0)[0][0]

    return float(score)

In [8]:
def sentiment_score(text):
    words = clean_text(text).split()

    pos = sum(1 for w in words if w in positive_words)
    neg = sum(1 for w in words if w in negative_words)

    total = pos + neg

    if total == 0:
        return 0

    return (pos - neg) / total

def predict_stance(claim, article_text):

    clean_claim = clean_text(claim)
    clean_article = clean_text(article_text)

    claim_seq = claim_tokenizer.texts_to_sequences([clean_claim])
    article_seq = article_tokenizer.texts_to_sequences([clean_article])

    claim_pad = pad_sequences(
        claim_seq,
        maxlen=MAX_CLAIM_LEN,
        padding="post",
        truncating="post"
    )

    article_pad = pad_sequences(
        article_seq,
        maxlen=MAX_ARTICLE_LEN,
        padding="post",
        truncating="post"
    )

    model_score = model_stance.predict(
        [claim_pad, article_pad],
        verbose=0
    )[0][0]

    text = clean_article

    strong_bantah_hit = sum(
        k in text for k in strong_bantah_keywords
    )

    weak_bantah_hit = sum(
        k in text for k in weak_bantah_keywords
    )

    strong_dukung_hit = sum(
        k in text for k in strong_dukung_keywords
    )

    weak_dukung_hit = sum(
        k in text for k in weak_dukung_keywords
    )


    sent_score = sentiment_score(text)

    boost = 0

    if strong_bantah_hit > 0:
        boost -= 0.45

    elif strong_dukung_hit > 0:
        boost += 0.35

    elif weak_bantah_hit > weak_dukung_hit:
        boost -= 0.15

    elif weak_dukung_hit > weak_bantah_hit:
        boost += 0.15

    # sentiment negatif
    if sent_score < -0.2:
        boost -= 0.10

    # sentiment positif
    elif sent_score > 0.2:
        boost += 0.10

    final_score = model_score + boost
    final_score = max(0, min(1, final_score))

    if strong_bantah_hit > 0:
        final_score = min(final_score, 0.25)

    elif strong_dukung_hit > 0:
        final_score = max(final_score, 0.75)

    elif weak_bantah_hit > weak_dukung_hit:
        final_score = min(final_score, 0.35)

    elif weak_dukung_hit > weak_bantah_hit:
        final_score = max(final_score, 0.55)

    if final_score < 0.4:
        label = "membantah"
    else:
        label = "mendukung"

    return label, float(final_score)

In [9]:
def get_similarity(headline, article_text):
    texts = [
        clean_text(headline),
        clean_text(article_text)
    ]

    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(texts)

    sim = cosine_similarity(
        tfidf[0:1],
        tfidf[1:2]
    )[0][0]

    return float(sim)

def is_trusted_source(source):
    source = str(source).lower().strip()

    all_pers_text = " ".join(
        pers.astype(str)
        .apply(lambda x: " ".join(x), axis=1)
        .str.lower()
        .tolist()
    )

    return source in all_pers_text

In [10]:
def analyze_news_claim(headline, max_articles=10):
    clickbait_score = predict_clickbait(headline)

    if clickbait_score >= THRESHOLD_CLICKBAIT:
        return {
            "headline": headline,
            "is_clickbait": True,
            "clickbait_score": clickbait_score,
            "verdict": "CLICKBAIT"
        }
    df_articles = search_google_news(headline, max_articles=max_articles)

    results = []

    for _, row in df_articles.iterrows():
        title = row["title"]
        content = row["content"]
        source = row["source"]

        article_text = title + " " + content

        article_clickbait_score = predict_clickbait(title)

        article_clickbait_label = (
            "yes" if article_clickbait_score >= THRESHOLD_CLICKBAIT else "no"
        )


        similarity = get_similarity(headline, article_text)

        stance_label, stance_model_score = predict_stance(
            headline,
            article_text
        )

        trusted = is_trusted_source(source)

        stance_risk = 1.0 - stance_model_score

        trusted_bantah_score = 1.0 if trusted and stance_label == "membantah" else 0.0
        trusted_dukung_score = 1.0 if trusted and stance_label == "mendukung" else 0.0

        article_hoax_score = (
            0.55 * stance_risk +
            0.30 * (1 - similarity) +
            0.15 * trusted_bantah_score
        )

        article_hoax_score -= 0.20 * trusted_dukung_score

        article_hoax_score = max(0, min(1, article_hoax_score))

        results.append({
            "title": title,
            "source": source,
            "url": row["url"],
            "published": row["published"],
            "similarity": round(similarity, 4),
            "stance_model_score": round(stance_model_score, 4),
            "stance": stance_label,
            "article_clickbait_score": round(article_clickbait_score, 4),
            "article_clickbait": article_clickbait_label,
            "article_hoax_score": round(article_hoax_score, 4),
            "trusted_source": trusted
        })

    df_result = pd.DataFrame(results)

    if len(df_result) == 0:
        return {
            "headline": headline,
            "message": "Tidak ditemukan artikel pembanding."
        }

    relevant_df = df_result[
        (df_result["article_clickbait"] == "no") &
        (df_result["similarity"] >= 0.15)
    ]

    if len(relevant_df) == 0:
      return {
          "headline": headline,
          "message": "Tidak ditemukan artikel non-clickbait yang relevan.",
          "articles": df_result
      }

    avg_similarity = relevant_df["similarity"].mean()
    avg_stance_model_score = relevant_df["stance_model_score"].mean()
    avg_article_hoax_score = relevant_df["article_hoax_score"].mean()

    membantah_count = (relevant_df["stance"] == "membantah").sum()
    mendukung_count = (relevant_df["stance"] == "mendukung").sum()

    trusted_bantah_ratio = (
        (relevant_df["trusted_source"] == True) &
        (relevant_df["stance"] == "membantah")
    ).mean()

    trusted_dukung_ratio = (
        (relevant_df["trusted_source"] == True) &
        (relevant_df["stance"] == "mendukung")
    ).mean()

    final_score = avg_article_hoax_score
    final_score = max(0, min(1, final_score))

    if final_score >= 0.60:
        verdict = "Kemungkinan Hoax"
    elif final_score >= 0.40:
        verdict = "Perlu Verifikasi Lanjutan"
    else:
        verdict = "Kemungkinan Asli"

    output = {
        "headline": headline,
        "verdict": verdict,
        "final_hoax_score": round(final_score, 4),
        "avg_similarity": round(avg_similarity, 4),
        "avg_stance_model_score": round(avg_stance_model_score, 4),
        "avg_article_hoax_score": round(avg_article_hoax_score, 4),
        "trusted_bantah_ratio": round(float(trusted_bantah_ratio), 4),
        "trusted_dukung_ratio": round(float(trusted_dukung_ratio), 4),
        "membantah_count": int(membantah_count),
        "mendukung_count": int(mendukung_count),
        "articles": df_result
    }

    return output

In [11]:
headline = "Nadiem Makarim didakwa pidana 18 tahun"

hasil = analyze_news_claim(headline, max_articles=5)

hasil["articles"]

,title,source,url,published,similarity,stance_model_score,stance,article_clickbait_score,article_clickbait,article_hoax_score,trusted_source
0,Nadiem Makarim Dituntut 18 Tahun Penjara dan U...,InvestorTrust,https://news.google.com/rss/articles/CBMi0AFBV...,"Wed, 13 May 2026 11:54:52 GMT",0.2797,0.7771,mendukung,0.1789,no,0.1387,True
1,Nadiem Makarim Dituntut 18 Tahun Penjara - det...,detikNews,https://news.google.com/rss/articles/CBMihwFBV...,"Wed, 13 May 2026 07:00:00 GMT",0.4503,0.7944,mendukung,0.0907,no,0.2780,False
2,JPU Tuntut Terdakwa Nadiem Makarim Divonis Pen...,Kejaksaan Republik Indonesia,https://news.google.com/rss/articles/CBMi7gFBV...,"Thu, 14 May 2026 07:00:00 GMT",0.2377,0.7500,mendukung,0.1865,no,0.3662,False
3,Nadiem Makarim Dituntut 18 Tahun Penjara dalam...,Kompas.com,https://news.google.com/rss/articles/CBMitwFBV...,"Wed, 13 May 2026 07:00:00 GMT",0.3360,0.7619,mendukung,0.1259,no,0.1301,True
4,Jaksa Tuntut Nadiem Makarim 18 Tahun Penjara -...,Bloomberg Technoz,https://news.google.com/rss/articles/CBMinAFBV...,"Wed, 13 May 2026 07:00:00 GMT",0.3808,0.5992,mendukung,0.1054,no,0.4062,False


Deploy

In [12]:
# FORMAT RESPONSE UNTUK BACKEND
def format_response_for_backend(hasil):
    if hasil.get("is_clickbait"):
        return {
            "skor_clickbait": round(
                hasil["clickbait_score"] * 100,
                2
            ),
            "verdict": "CLICKBAIT",
            "keyakinan": "tinggi",
            "berita_terkait": []
        }
    df_articles = hasil["articles"].copy()

    df_articles = df_articles[
        (df_articles["article_clickbait"] == "no") &
        (df_articles["similarity"] >= 0.15)
    ]

    if "final_hoax_score" not in hasil:

        berita_terkait = []

        return {
            "skor_hoax": None,
            "verdict": "TIDAK DAPAT DIANALISIS",
            "keyakinan": "rendah",
            "message": hasil.get("message"),
            "berita_terkait": berita_terkait
        }

    berita_terkait = []

    for i, (_, row) in enumerate(df_articles.iterrows(), start=1):
        berita_terkait.append({
            "id": i,
            "judul": row["title"],
            "sumber": row["source"],
            "similarity": None if pd.isna(row["similarity"]) else round(row["similarity"] * 100, 2),
            "status": None if pd.isna(row["stance"]) else str(row["stance"]).capitalize(),
            "trusted_source": None if pd.isna(row["trusted_source"]) else bool(row["trusted_source"]),
            "skor_hoax_artikel": None if pd.isna(row["article_hoax_score"]) else round(row["article_hoax_score"] * 100, 2),
            "url": row["url"]
        })

    # hitungan skor hoax dijadiin rentang 0-100
    skor_hoax = 0

    total_konfirmasi = hasil["membantah_count"] + hasil["mendukung_count"]

    if total_konfirmasi > 0:
        rasio_bantah = hasil["membantah_count"] / total_konfirmasi
    else:
        rasio_bantah = 0

    skor_hoax += rasio_bantah * 45

    skor_hoax += hasil["avg_article_hoax_score"] * 30

    skor_hoax += (1 - hasil["avg_similarity"]) * 10

    skor_hoax += hasil["trusted_bantah_ratio"] * 15

    skor_hoax -= hasil["trusted_dukung_ratio"] * 10

    skor_hoax = max(0, min(100, skor_hoax))
    skor_hoax = round(skor_hoax)

    # verdict
    if skor_hoax >= 75:
        verdict = "SANGAT MUNGKIN HOAX"

    elif skor_hoax >= 55:
        verdict = "KEMUNGKINAN HOAX"

    elif skor_hoax >= 40:
        verdict = "TIDAK PASTI (PERLU VERIFIKASI)"

    elif skor_hoax >= 20:
        verdict = "KEMUNGKINAN BENAR"

    else:
        verdict = "SANGAT MUNGKIN BENAR"

    # Keyakinan
    if skor_hoax >= 75 or skor_hoax <= 19:
        keyakinan = "tinggi"
    elif skor_hoax >= 55 or skor_hoax <= 39:
        keyakinan = "sedang"
    else:
        keyakinan = "rendah"

    # Analisis semantik
    analisis_semantik = []

    # cek fakta
    if hasil["trusted_bantah_ratio"] > 0:
        cek_fakta_resmi = "Ditemukan sumber terpercaya yang membantah"
    elif hasil["trusted_dukung_ratio"] > 0:
        cek_fakta_resmi = "Ditemukan sumber terpercaya yang mendukung"
    else:
        cek_fakta_resmi = "Tidak ada cek fakta resmi ditemukan"

    response = {
        "skor_hoax": skor_hoax,
        "verdict": verdict,
        "keyakinan": keyakinan,
        "komponen_analisis": {
            "rasio_konfirmasi": {
                "membantah": hasil["membantah_count"],
                "mendukung": hasil["mendukung_count"]
            },
            "cek_fakta_resmi": cek_fakta_resmi,
            "analisis_semantik": analisis_semantik,
            "jangkauan_penelusuran": len(df_articles)
        },
        "berita_terkait": berita_terkait
    }

    return response

In [15]:
headline = "Jakarta kembali menjadi ibukota"

hasil = analyze_news_claim(headline, max_articles=5)

response_backend = format_response_for_backend(hasil)

response_backend

{'skor_hoax': 4,
 'verdict': 'SANGAT MUNGKIN BENAR',
 'keyakinan': 'tinggi',
 'komponen_analisis': {'rasio_konfirmasi': {'membantah': 0, 'mendukung': 1},
  'cek_fakta_resmi': 'Ditemukan sumber terpercaya yang mendukung',
  'analisis_semantik': [],
  'jangkauan_penelusuran': 1},
 'berita_terkait': [{'id': 1,
   'judul': 'Jakarta Kembali Catat Kualitas Udara Terburuk, Warga Diimbau Waspada - Minanews.net',
   'sumber': 'Minanews.net',
   'similarity': 18.44,
   'status': 'Mendukung',
   'trusted_source': True,
   'skor_hoax_artikel': 20.83,
   'url': 'https://news.google.com/rss/articles/CBMikwFBVV95cUxQT2IxbXoyM3JNSVNXOE5tTDVFWVdYNGttMlRZZlQ4U3p6ekRCM1JOTzQzbS1tZ0dnLXhmX3BqTXdvTUxzeER4UURmbW1hellvS0M5Y3JCSkJMMjh6bFZxOV9pRmNvMXVKaW1sUEFja3JfZDFTTGhFRHFQOVh2NFRpRXhnVEZRMmNGaG5QOEx0d0JxZjA?oc=5'}]}

In [ ]:
!pip install flask pyngrok

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok

In [ ]:
app = Flask(__name__)

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Hoax Detection API is running",
        "endpoint": "/predict",
        "method": "POST"
    })

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        if data is None:
            return jsonify({
                "error": "Request body harus JSON"
            }), 400

        headline = data.get("headline", "")

        if headline.strip() == "":
            return jsonify({
                "error": "headline kosong"
            }), 400

        max_articles = data.get("max_articles", 5)

        hasil = analyze_news_claim(
            headline=headline,
            max_articles=max_articles
        )

        response_backend = format_response_for_backend(hasil)

        return jsonify(response_backend), 200

    except Exception as e:
        return jsonify({
            "error": str(e
            )
        }), 500

ngrok.set_auth_token("3ECbLy7i35nY143a8ARk3RMBQYu_7zEWVpLcJ8etCBJetzMPV")
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://swooned-pregame-saline.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/May/2026 14:57:42] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/May/2026 15:06:32] "POST /predict HTTP/1.1" 200 -
